### **✏️ Exercises**
- Use LangChain's docling integration for document ingestion
- Rewrite the whole pipeline using LangChain

In [39]:
#1.Use LangChain's docling integration for document ingestion
%pip install -q langchain-docling

In [1]:
from langchain_docling import DoclingLoader

In [2]:
loader = DoclingLoader(
    file_path="https://raw.githubusercontent.com/data-engineer-portfolio/AI_hands_on/main/insurance_rag_knowledge_base.pdf"
)

documents = loader.load()

print("Number of documents:", len(documents))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[INFO] 2026-06-10 13:16:00,996 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-10 13:16:01,007 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-10 13:16:01,067 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-10 13:16:01,069 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Number of documents: 109


In [3]:
print(documents[0].page_content[:1000])

RAG Pipeline Document
Version 1.0 · Domain: Insurance · June 2026
Coverage: Policies · Claims · Underwriting · Regulations · Glossary
1, Topic = Types of Insurance Policies. 2, Topic = Insurance Claims Process. 3, Topic = Insurance Underwriting. 4, Topic = Policy Terms & Conditions. 5, Topic = Regulations & Compliance. 6, Topic = Common Terms Glossary. 7, Topic = Premium Factors & Discounts. 8, Topic = Insurance Industry Overview


In [ ]:
#2.Rewrite the whole pipeline using LangChain

In [4]:
%pip install -q \
langchain \
langchain-community \
langchain-qdrant \
langchain-groq \
sentence-transformers \
qdrant-client

In [5]:
#Installing dependencies
from langchain_docling import DoclingLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore

from langchain_core.embeddings import Embeddings

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [6]:
# Loading PDF
loader = DoclingLoader(
    file_path="https://raw.githubusercontent.com/data-engineer-portfolio/AI_hands_on/main/insurance_rag_knowledge_base.pdf"
)

documents = loader.load()

print(len(documents))

[INFO] 2026-06-10 13:18:07,946 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-10 13:18:07,948 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-10 13:18:08,005 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-10 13:18:08,006 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-10 13:18:08,256 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-10 13:18:08,257 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-10 13:18:08,263 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-06-10 13:18:08,264 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-06-10 13:18:08,363 [Ra

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

109


In [7]:
#4. Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 111


In [8]:
#5. Create embedding wrapper
class SentenceTransformerEmbeddings(Embeddings):

    def __init__(self):
        self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"
        )

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode(text).tolist()

In [9]:
#6. Initialize embeddings
embeddings = SentenceTransformerEmbeddings()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
#7.Create qdrant vector store
# client = QdrantClient(":memory:") # Removed as QdrantVectorStore will manage its internal client

vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:", # Pass the location for internal client creation
    collection_name="insurance_docs"
)

print("Vector DB Created")

Vector DB Created


In [11]:
#8. Create retreiver
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [12]:
#9. Test Retrieval
docs = retriever.invoke(
    "What is the leave policy?"
)

for doc in docs:
    print(doc.page_content[:500])
    print("="*50)

COBRA:
A federal law that allows employees to continue employer-sponsored health insurance coverage after leaving their job.
Policy:
The written contract between the insured and the insurer that outlines the terms and conditions of coverage.
Lapse:
The termination of an insurance policy due to non-payment of premiums.


In [13]:
#10.Configure Groq API Key
import os

os.environ["GROQ_API_KEY"] = "Your Groq Key"

In [14]:
#11. Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [15]:
# 12. Prompt
prompt = ChatPromptTemplate.from_template(
"""
You are an Insurance assistant.

Answer ONLY from the supplied context.

Context:
{context}

Question:
{question}
"""
)

In [16]:
#13. Helper Function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [17]:
#14. Build RAG Chain
chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [18]:
#15. Ask Question
response = chain.invoke(
    "How many claims employees are entitled to?"
)

print(response)

The context does not specify the number of claims employees are entitled to. It only defines a claim and explains the claims assignment process, and mentions Workers' Compensation as a type of insurance.


In [19]:
#16.Ask another Question
response = chain.invoke(
    "What happens if my insurance claim is denied?"
)

print(response)

The supplied context does not provide information on what happens if an insurance claim is denied.


In [20]:
#17. Ask another Questio
response=chain.invoke("What is an endorsement or rider in an insurance policy?")
print(response)

An endorsement (also called a rider) is an amendment to an insurance policy that modifies its terms, adds coverage, removes coverage, or clarifies existing provisions.
